In [1]:
import polars as pl
from pathlib import Path
import climato.spikes.daily_wide_long as dwl

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
print("cwd:", Path.cwd())
print("module:", dwl.__file__)
print("DATA_DIR:", dwl.DATA_DIR)
print("DATA_DIR exists:", dwl.DATA_DIR.exists())

print("RTV matches:")
print(list(dwl.DATA_DIR.glob(dwl.RR_T_VENT_PATTERN)))

print("AP matches:")
print(list(dwl.DATA_DIR.glob(dwl.AUTRES_PARAMETRES_PATTERN)))

import sys

print(sys.executable)

cwd: /home/antoine/projects/personal/climato-lab/src/climato/spikes
module: /home/antoine/projects/personal/climato-lab/src/climato/spikes/daily_wide_long.py
DATA_DIR: /home/antoine/projects/personal/climato-lab/data/raw/meteo_france/daily
DATA_DIR exists: True
RTV matches:
[PosixPath('/home/antoine/projects/personal/climato-lab/data/raw/meteo_france/daily/Q_79_avant-1949_RR-T-Vent.csv'), PosixPath('/home/antoine/projects/personal/climato-lab/data/raw/meteo_france/daily/Q_79_latest-2025-2026_RR-T-Vent.csv'), PosixPath('/home/antoine/projects/personal/climato-lab/data/raw/meteo_france/daily/Q_79_previous-1950-2024_RR-T-Vent.csv')]
AP matches:
[PosixPath('/home/antoine/projects/personal/climato-lab/data/raw/meteo_france/daily/Q_79_previous-1950-2024_autres-parametres.csv'), PosixPath('/home/antoine/projects/personal/climato-lab/data/raw/meteo_france/daily/Q_79_avant-1949_autres-parametres.csv'), PosixPath('/home/antoine/projects/personal/climato-lab/data/raw/meteo_france/daily/Q_79_lates

The CSV source data files contain daily observations of meteorological parameters. Each parameter corresponds to a column in the files.
Each parameter value has a corresponding quality code. In the data files, quality codes are listed in a separate column. For a given parameter, the quality code is in the column immediately to the right of the column containing the parameter itself.

In [4]:
lf_rtv_source = dwl.scan_daily_rr_t_vent()
lf_rtv_wide_core = dwl.build_lf_rtv_wide_core(lf_rtv_source)
lf_rtv_wide_all = dwl.build_lf_rtv_wide_all(lf_rtv_source)

lf_ap_source = dwl.scan_daily_autres_parametres()
lf_ap_wide = dwl.build_lf_ap_wide(lf_ap_source)

## 2. Source data understanding

### 2.1. Corpus

The spike is currently limited to the daily MDGF data for department 79, the observed parameter coverage here is not assumed to be representative of the coverage at the national level, when all data are included.

In [5]:
families = {
    "RR-T-Vent": {
        "lf": lf_rtv_wide_all,
        "params": dwl.RR_T_VENT_ALL_PARAMS,
    },
    "autres-parametres": {
        "lf": lf_ap_wide,
        "params": dwl.AUTRES_PARAMETRES_ALL_PARAMS,
    },
}

family_summaries: list[pl.LazyFrame] = []
for family_name, family_dict in families.items():
    lf: pl.LazyFrame = family_dict["lf"]
    family_summaries.append(
        lf
        .select(
            source_family=pl.lit(family_name),
            files=pl.col("source_file").n_unique(),
            rows=pl.len(),
            stations=pl.col("station_name").n_unique(),
            parameters=len(family_dict["params"]),
            min_date=pl.col("obs_date").min(),
            max_date=pl.col("obs_date").max(),
        ),
    )

print(pl.concat(family_summaries).collect())

shape: (2, 7)
┌───────────────────┬───────┬─────────┬──────────┬────────────┬────────────┬────────────┐
│ source_family     ┆ files ┆ rows    ┆ stations ┆ parameters ┆ min_date   ┆ max_date   │
│ ---               ┆ ---   ┆ ---     ┆ ---      ┆ ---        ┆ ---        ┆ ---        │
│ str               ┆ u32   ┆ u32     ┆ u32      ┆ i32        ┆ date       ┆ date       │
╞═══════════════════╪═══════╪═════════╪══════════╪════════════╪════════════╪════════════╡
│ RR-T-Vent         ┆ 3     ┆ 1172316 ┆ 117      ┆ 26         ┆ 1865-07-01 ┆ 2026-08-12 │
│ autres-parametres ┆ 3     ┆ 565558  ┆ 70       ┆ 41         ┆ 1867-01-02 ┆ 2026-08-16 │
└───────────────────┴───────┴─────────┴──────────┴────────────┴────────────┴────────────┘


### 2.2. Parameter completeness

**TODO**
Parameter completeness -> try to identify groups based on count_pct

In [6]:
print("RR-T-Vent:")
with pl.Config(tbl_cols=20, tbl_rows=-1, tbl_width_chars=170):
    print(dwl.custom_describe(lf_rtv_wide_all))

RR-T-Vent:
shape: (58, 5)
┌──────────────┬─────────┬────────────┬───────────┬────────────────┐
│ column       ┆ count   ┆ null_count ┆ count_pct ┆ null_count_pct │
│ ---          ┆ ---     ┆ ---        ┆ ---       ┆ ---            │
│ str          ┆ u32     ┆ u32        ┆ f64       ┆ f64            │
╞══════════════╪═════════╪════════════╪═══════════╪════════════════╡
│ station_id   ┆ 1172316 ┆ 0          ┆ 100.0     ┆ 0.0            │
│ station_name ┆ 1172316 ┆ 0          ┆ 100.0     ┆ 0.0            │
│ obs_date     ┆ 1172316 ┆ 0          ┆ 100.0     ┆ 0.0            │
│ source_file  ┆ 1172316 ┆ 0          ┆ 100.0     ┆ 0.0            │
│ rr           ┆ 1169046 ┆ 3270       ┆ 99.72     ┆ 0.28           │
│ qrr          ┆ 1169115 ┆ 3201       ┆ 99.73     ┆ 0.27           │
│ tn           ┆ 437999  ┆ 734317     ┆ 37.36     ┆ 62.64          │
│ qtn          ┆ 439710  ┆ 732606     ┆ 37.51     ┆ 62.49          │
│ htn          ┆ 153736  ┆ 1018580    ┆ 13.11     ┆ 86.89          │
│ qhtn  

In [7]:
print("autres-parametres:")
with pl.Config(tbl_cols=20, tbl_rows=-1, tbl_width_chars=170):
    print(dwl.custom_describe(lf_ap_wide))

autres-parametres:
shape: (86, 5)
┌──────────────┬────────┬────────────┬───────────┬────────────────┐
│ column       ┆ count  ┆ null_count ┆ count_pct ┆ null_count_pct │
│ ---          ┆ ---    ┆ ---        ┆ ---       ┆ ---            │
│ str          ┆ u32    ┆ u32        ┆ f64       ┆ f64            │
╞══════════════╪════════╪════════════╪═══════════╪════════════════╡
│ station_id   ┆ 565558 ┆ 0          ┆ 100.0     ┆ 0.0            │
│ station_name ┆ 565558 ┆ 0          ┆ 100.0     ┆ 0.0            │
│ obs_date     ┆ 565558 ┆ 0          ┆ 100.0     ┆ 0.0            │
│ source_file  ┆ 565558 ┆ 0          ┆ 100.0     ┆ 0.0            │
│ dhumec       ┆ 7945   ┆ 557613     ┆ 1.4       ┆ 98.6           │
│ qdhumec      ┆ 8096   ┆ 557462     ┆ 1.43      ┆ 98.57          │
│ pmerm        ┆ 14801  ┆ 550757     ┆ 2.62      ┆ 97.38          │
│ qpmerm       ┆ 14801  ┆ 550757     ┆ 2.62      ┆ 97.38          │
│ pmermin      ┆ 6819   ┆ 558739     ┆ 1.21      ┆ 98.79          │
│ qpmermin    

### 2.3 Measurement density & recurring parameter signatures

#### 2.3.1. `RR-T-Vent`

**TODO**

The 1,172,316 wide rows actually contain 4,242,670 (non-null) measurements.  
Therefore:
- Each wide row has on average only approximately **3.62** measurements for the 26 parameters. 
- Measurement density is **13.9%** (= 4,242,670 / (1,172,316 * 26)).

In [8]:
lf_rtv_wide_all = dwl.add_measure_count_lf(lf_rtv_wide_all, dwl.RR_T_VENT_ALL_PARAMS, "all_measure_count")

summary = (
    lf_rtv_wide_all
    .select(
        n_rows=pl.len(),
        n_measures=pl.col("all_measure_count").sum(),
    )
    .with_columns(
        mean_measures_per_wide_row=pl.col("n_measures") / pl.col("n_rows"),
        measure_density=pl.col("n_measures") / (pl.col("n_rows") * len(dwl.RR_T_VENT_ALL_PARAMS)),
    )
    .collect()
)

print("RR-T-Vent all parameters:")
print(summary)

RR-T-Vent all parameters:
shape: (1, 4)
┌─────────┬────────────┬────────────────────────────┬─────────────────┐
│ n_rows  ┆ n_measures ┆ mean_measures_per_wide_row ┆ measure_density │
│ ---     ┆ ---        ┆ ---                        ┆ ---             │
│ u32     ┆ u32        ┆ f64                        ┆ f64             │
╞═════════╪════════════╪════════════════════════════╪═════════════════╡
│ 1172316 ┆ 4242670    ┆ 3.61905                    ┆ 0.139194        │
└─────────┴────────────┴────────────────────────────┴─────────────────┘


Below is the breakdown of wide rows by number of measured (non-null) parameters.

In [9]:
with pl.Config(tbl_cols=20, tbl_rows=100):
    print(
        lf_rtv_wide_all
        .group_by("all_measure_count")
        .len(name="count")
        .with_columns(
            pct=pl.col("count") / pl.col("count").sum() * 100,
        )
        .sort("all_measure_count")
        .collect()
    )

shape: (23, 3)
┌───────────────────┬────────┬───────────┐
│ all_measure_count ┆ count  ┆ pct       │
│ ---               ┆ ---    ┆ ---       │
│ u32               ┆ u32    ┆ f64       │
╞═══════════════════╪════════╪═══════════╡
│ 0                 ┆ 427    ┆ 0.036424  │
│ 1                 ┆ 733810 ┆ 62.594898 │
│ 2                 ┆ 69     ┆ 0.005886  │
│ 3                 ┆ 4      ┆ 0.000341  │
│ 4                 ┆ 861    ┆ 0.073444  │
│ 5                 ┆ 280919 ┆ 23.962737 │
│ 6                 ┆ 1023   ┆ 0.087263  │
│ 7                 ┆ 731    ┆ 0.062355  │
│ 8                 ┆ 5882   ┆ 0.501742  │
│ 9                 ┆ 81105  ┆ 6.918356  │
│ 10                ┆ 206    ┆ 0.017572  │
│ 11                ┆ 153    ┆ 0.013051  │
│ 12                ┆ 42     ┆ 0.003583  │
│ 13                ┆ 49     ┆ 0.00418   │
│ 14                ┆ 28     ┆ 0.002388  │
│ 15                ┆ 91     ┆ 0.007762  │
│ 16                ┆ 2498   ┆ 0.213082  │
│ 17                ┆ 466    ┆ 0.03975 

Below is the breakdown of wide rows by the exact combination of measured parameters (signature).  
Of the 344 signature profiles observed on all `RR-T-Vent` rows:
- the most frequently observed accounts for approximately **62.6%** of all `RR-T-Vent` rows (rows with only `RR`)
- the 3 most frequently observed account for more than **93.4%** of all rows
- the 10 most frequently observed account for approximately **99.0%** of all rows 

In [10]:
lf_rtv_wide_all = lf_rtv_wide_all.with_columns(
    dwl.measure_signature_expr(dwl.RR_T_VENT_ALL_PARAMS)
    .alias("measure_signature")
)

patterns_rtv_all = (
    lf_rtv_wide_all
    .group_by(
        "all_measure_count",
        "measure_signature",
    )
    .agg(
        count=pl.len(),
    )
    .with_columns(
        pct=pl.col("count") / pl.col("count").sum() * 100
    )
    .sort("count", descending=True)
    .collect()
)

patterns_rtv_all_head = patterns_rtv_all.head(30)
print(f"patterns_rtv_all.shape: {patterns_rtv_all.shape}")
print(f"patterns_rtv_all sum pct: {patterns_rtv_all.select("pct").sum().item()}")
with pl.Config(tbl_cols=20, tbl_rows=100, fmt_str_lengths=300):   
    print(patterns_rtv_all_head)

patterns_rtv_all.shape: (344, 4)
patterns_rtv_all sum pct: 100.00000000000006
shape: (30, 4)
┌───────────────────┬─────────────────────────────────────────────────────────┬────────┬───────────┐
│ all_measure_count ┆ measure_signature                                       ┆ count  ┆ pct       │
│ ---               ┆ ---                                                     ┆ ---    ┆ ---       │
│ u32               ┆ str                                                     ┆ u32    ┆ f64       │
╞═══════════════════╪═════════════════════════════════════════════════════════╪════════╪═══════════╡
│ 1                 ┆ RR                                                      ┆ 733800 ┆ 62.594045 │
│ 5                 ┆ RR,TN,TX,TNTXM,TAMPLI                                   ┆ 280908 ┆ 23.961799 │
│ 9                 ┆ RR,TN,HTN,TX,HTX,TM,TNTXM,TAMPLI,DG                     ┆ 81036  ┆ 6.912471  │
│ 19                ┆ RR,TN,HTN,TX,HTX,TM,TNTXM,TAMPLI,DG,FFM,FXY,DXY,HXY,FXI ┆ 23138  ┆ 1.9737    

#### 2.3.2. `autres-parametres`

**TODO**

The 565,558 wide rows actually contain 2,148,378 parameter measurements.  
Therefore:
- Each wide row has on average only approximately **3.80** measurements for the 41 parameters. 
- Measurement density is **9.3%** (= 2,148,378 / (565,558 * 41)).

In [11]:
lf_ap_wide = dwl.add_measure_count_lf(lf_ap_wide, dwl.AUTRES_PARAMETRES_ALL_PARAMS, "all_measure_count")

summary = (
    lf_ap_wide
    .select(
        n_rows=pl.len(),
        n_measures=pl.col("all_measure_count").sum(),
    )
    .with_columns(
        mean_measures_per_wide_row=pl.col("n_measures") / pl.col("n_rows"),
        measure_density=pl.col("n_measures") / (pl.col("n_rows") * len(dwl.AUTRES_PARAMETRES_ALL_PARAMS)),
    )
    .collect()
)

print("autres-parameters all parameters:")
print(summary)

autres-parameters all parameters:
shape: (1, 4)
┌────────┬────────────┬────────────────────────────┬─────────────────┐
│ n_rows ┆ n_measures ┆ mean_measures_per_wide_row ┆ measure_density │
│ ---    ┆ ---        ┆ ---                        ┆ ---             │
│ u32    ┆ u32        ┆ f64                        ┆ f64             │
╞════════╪════════════╪════════════════════════════╪═════════════════╡
│ 565558 ┆ 2148378    ┆ 3.798687                   ┆ 0.092651        │
└────────┴────────────┴────────────────────────────┴─────────────────┘


Below is the breakdown of wide rows by number of measured (non-null) parameters (signature length).

In [12]:
with pl.Config(tbl_cols=20, tbl_rows=100):
    print(
        lf_ap_wide
        .group_by("all_measure_count")
        .len(name="count")
        .with_columns(
            pct=pl.col("count") / pl.col("count").sum() * 100,
        )
        .sort("all_measure_count")
        .collect()
    )

shape: (30, 3)
┌───────────────────┬────────┬───────────┐
│ all_measure_count ┆ count  ┆ pct       │
│ ---               ┆ ---    ┆ ---       │
│ u32               ┆ u32    ┆ f64       │
╞═══════════════════╪════════╪═══════════╡
│ 0                 ┆ 370    ┆ 0.065422  │
│ 1                 ┆ 122836 ┆ 21.719435 │
│ 2                 ┆ 21626  ┆ 3.823834  │
│ 3                 ┆ 278177 ┆ 49.18629  │
│ 4                 ┆ 71335  ┆ 12.613207 │
│ 5                 ┆ 1198   ┆ 0.211826  │
│ 6                 ┆ 1330   ┆ 0.235166  │
│ 7                 ┆ 1741   ┆ 0.307838  │
│ 8                 ┆ 794    ┆ 0.140392  │
│ 9                 ┆ 39317  ┆ 6.951895  │
│ 10                ┆ 306    ┆ 0.054106  │
│ 11                ┆ 7693   ┆ 1.36025   │
│ 12                ┆ 1256   ┆ 0.222082  │
│ 13                ┆ 2783   ┆ 0.49208   │
│ 14                ┆ 78     ┆ 0.013792  │
│ 15                ┆ 372    ┆ 0.065776  │
│ 16                ┆ 255    ┆ 0.045088  │
│ 17                ┆ 671    ┆ 0.118644

Below is the breakdown of wide rows by the exact combination of measured parameters (signature).  
Of the 635 signature profiles observed on all `autres-parametres` rows:
- the most frequently observed accounts for more than **47.8%** of all `autres-parametres` rows (signature length 3: rows with only `BROU`, `ORAG` and `GRELE`)
- the 3 most frequently observed account for approximately **79.9%** of all rows (signature length <= 4)
- the 10 most frequently observed account for more than **92.5%** of all rows (signature length <= 11)

In [13]:
lf_ap_wide = lf_ap_wide.with_columns(
    dwl.measure_signature_expr(dwl.AUTRES_PARAMETRES_ALL_PARAMS)
    .alias("measure_signature")
)

patterns_ap = (
    lf_ap_wide
    .group_by(
        "all_measure_count",
        "measure_signature",
    )
    .agg(
        count=pl.len(),
    )
    .with_columns(
        pct=pl.col("count") / pl.col("count").sum() * 100
    )
    .sort("count", descending=True)
    .collect()
)

patterns_ap_head = patterns_ap.head(30)
print(f"patterns_ap.shape: {patterns_ap.shape}")
print(f"patterns_ap, sum pct: {patterns_ap.select("pct").sum().item()}")
with pl.Config(tbl_cols=20, tbl_rows=100, fmt_str_lengths=300):   
    print(patterns_ap_head)

patterns_ap.shape: (635, 4)
patterns_ap, sum pct: 99.99999999999996
shape: (30, 4)
┌───────────────────┬─────────────────────────────────────────────────────────┬────────┬───────────┐
│ all_measure_count ┆ measure_signature                                       ┆ count  ┆ pct       │
│ ---               ┆ ---                                                     ┆ ---    ┆ ---       │
│ u32               ┆ str                                                     ┆ u32    ┆ f64       │
╞═══════════════════╪═════════════════════════════════════════════════════════╪════════╪═══════════╡
│ 3                 ┆ BROU,ORAG,GRELE                                         ┆ 270391 ┆ 47.809597 │
│ 1                 ┆ ETPGRILLE                                               ┆ 114089 ┆ 20.17282  │
│ 4                 ┆ ETPGRILLE,BROU,ORAG,GRELE                               ┆ 67175  ┆ 11.87765  │
│ 9                 ┆ UN,HUN,UX,HUX,UM,DHUMI40,DHUMI80,TSVM,ETPGRILLE         ┆ 34402  ┆ 6.082842  │
│ 2     

### DELETE 2.4 Recurring parameter signatures

#### 2.4.1 `RR-T-Vent`

##### 2.4.1.1. Core parameters

If we consider only 5 core parameters (`RR`, `TN`, `TX`, `TM`, `TNTXM`) out of the 26 in the `RR-T-Vent` files, we can observe that of the 1,172,316 lines:
- 62.60% have 1 measurement (most frequently `RR`)
- 24.25% have 4 measurements (most frequently the combination of `RR`, `TN`, `TX`, `TNTXM`)
- 13.03% have 5 measurements (all 5 core parameters)  

They account for virtually all of the DataFrame's lines (99.88%).

We have the following statistics:
- Number of wide rows: 1,172,316
- Expected number of long (core) rows (after unpivoting): 2,637,628
- Mean number of (core) measures per wide row: 2.2499
- (Core) measurement density: 45.00% (= 2.2499 / 5)

In [ ]:
lf_rtv_wide_core = lf_rtv_wide_core.with_columns(
    dwl.measure_signature_expr(dwl.RR_T_VENT_CORE_PARAMS)
    .alias("measure_signature")
)

patterns_rtv_core = (
    lf_rtv_wide_core
    .group_by(
        "core_measure_count",
        "measure_signature",
    )
    .agg(
        count=pl.len(),
    )
    .with_columns(
        pct=pl.col("count") / pl.col("count").sum() * 100
    )
    .sort("count", descending=True)
    .collect()
)

with pl.Config(tbl_cols=20, tbl_rows=100, fmt_str_lengths=300):   
    print(patterns_rtv_core.head(30))

shape: (14, 4)
┌────────────────────┬───────────────────┬────────┬───────────┐
│ core_measure_count ┆ measure_signature ┆ count  ┆ pct       │
│ ---                ┆ ---               ┆ ---    ┆ ---       │
│ u32                ┆ str               ┆ u32    ┆ f64       │
╞════════════════════╪═══════════════════╪════════╪═══════════╡
│ 1                  ┆ RR                ┆ 733846 ┆ 62.597968 │
│ 4                  ┆ RR,TN,TX,TNTXM    ┆ 282433 ┆ 24.091883 │
│ 5                  ┆ RR,TN,TX,TM,TNTXM ┆ 152733 ┆ 13.028313 │
│ 4                  ┆ TN,TX,TM,TNTXM    ┆ 1871   ┆ 0.159599  │
│ 3                  ┆ TN,TX,TNTXM       ┆ 934    ┆ 0.079671  │
│ 0                  ┆                   ┆ 442    ┆ 0.037703  │
│ 2                  ┆ RR,TN             ┆ 16     ┆ 0.001365  │
│ 1                  ┆ TX                ┆ 14     ┆ 0.001194  │
│ 2                  ┆ RR,TX             ┆ 9      ┆ 0.000768  │
│ 3                  ┆ RR,TN,TM          ┆ 7      ┆ 0.000597  │
│ 1                  ┆ TN

##### 2.4.1.2. All `RR-T-Vent` parameters

#### 2.4.2. `autres-parametres`

### 2.4 Value / quality consistency

#### 2.4.1. `RR-T-Vent`

##### 2.4.1.1. Audit of parameter / quality code pairs

In [ ]:
rtv_all_quality_audit = dwl.quality_pair_audit_lf(lf_rtv_wide_all, dwl.RR_T_VENT_ALL_PARAMS)

with pl.Config(tbl_rows=100):
    print(rtv_all_quality_audit)

shape: (52, 2)
┌──────────────────────────────┬───────┐
│ line_type                    ┆ count │
│ ---                          ┆ ---   │
│ str                          ┆ u32   │
╞══════════════════════════════╪═══════╡
│ RR_value_without_quality     ┆ 0     │
│ RR_quality_without_value     ┆ 69    │
│ TN_value_without_quality     ┆ 0     │
│ TN_quality_without_value     ┆ 1711  │
│ HTN_value_without_quality    ┆ 0     │
│ HTN_quality_without_value    ┆ 2066  │
│ TX_value_without_quality     ┆ 0     │
│ TX_quality_without_value     ┆ 1705  │
│ HTX_value_without_quality    ┆ 0     │
│ HTX_quality_without_value    ┆ 2082  │
│ TM_value_without_quality     ┆ 0     │
│ TM_quality_without_value     ┆ 1714  │
│ TNTXM_value_without_quality  ┆ 0     │
│ TNTXM_quality_without_value  ┆ 1     │
│ TAMPLI_value_without_quality ┆ 0     │
│ TAMPLI_quality_without_value ┆ 1     │
│ TNSOL_value_without_quality  ┆ 0     │
│ TNSOL_quality_without_value  ┆ 24    │
│ TN50_value_without_quality   ┆ 0     │
│

In [14]:
all_quality_pairs_metadata, all_quality_pairs_expressions = dwl.quality_pair_audit_lf_2(lf=None, params=dwl.RR_T_VENT_ALL_PARAMS)
all_quality_pairs_counts = (
    lf_rtv_wide_all
    .select(all_quality_pairs_expressions)
    .collect()
)
print(all_quality_pairs_counts)

shape: (1, 52)
┌──────────┬──────────┬──────────┬──────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ metric_0 ┆ metric_1 ┆ metric_2 ┆ metric_3 ┆ … ┆ metric_48 ┆ metric_49 ┆ metric_50 ┆ metric_51 │
│ ---      ┆ ---      ┆ ---      ┆ ---      ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ u32      ┆ u32      ┆ u32      ┆ u32      ┆   ┆ u32       ┆ u32       ┆ u32       ┆ u32       │
╞══════════╪══════════╪══════════╪══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 0        ┆ 69       ┆ 0        ┆ 1711     ┆ … ┆ 0         ┆ 17        ┆ 0         ┆ 1644      │
└──────────┴──────────┴──────────┴──────────┴───┴───────────┴───────────┴───────────┴───────────┘


In [15]:
all_quality_pairs_counts_long = all_quality_pairs_counts.unpivot(
    variable_name="metric_id",
    value_name="count",
)
print(all_quality_pairs_counts_long)

shape: (52, 2)
┌───────────┬───────┐
│ metric_id ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ metric_0  ┆ 0     │
│ metric_1  ┆ 69    │
│ metric_2  ┆ 0     │
│ metric_3  ┆ 1711  │
│ metric_4  ┆ 0     │
│ …         ┆ …     │
│ metric_47 ┆ 17    │
│ metric_48 ┆ 0     │
│ metric_49 ┆ 17    │
│ metric_50 ┆ 0     │
│ metric_51 ┆ 1644  │
└───────────┴───────┘


In [16]:
all_quality_pairs_metadata_df = pl.DataFrame(all_quality_pairs_metadata)
print(all_quality_pairs_metadata_df)

shape: (52, 3)
┌───────────┬───────────────────────┬───────┐
│ metric_id ┆ pair_type             ┆ param │
│ ---       ┆ ---                   ┆ ---   │
│ str       ┆ str                   ┆ str   │
╞═══════════╪═══════════════════════╪═══════╡
│ metric_0  ┆ value_without_quality ┆ RR    │
│ metric_1  ┆ quality_without_value ┆ RR    │
│ metric_2  ┆ value_without_quality ┆ TN    │
│ metric_3  ┆ quality_without_value ┆ TN    │
│ metric_4  ┆ value_without_quality ┆ HTN   │
│ …         ┆ …                     ┆ …     │
│ metric_47 ┆ quality_without_value ┆ DXI3S │
│ metric_48 ┆ value_without_quality ┆ HXI3S │
│ metric_49 ┆ quality_without_value ┆ HXI3S │
│ metric_50 ┆ value_without_quality ┆ DRR   │
│ metric_51 ┆ quality_without_value ┆ DRR   │
└───────────┴───────────────────────┴───────┘


In [18]:
all_quality_pairs_rtv_audit = (
    all_quality_pairs_counts_long
    .join(
        all_quality_pairs_metadata_df,
        how="left",
        on="metric_id",
    )
)
print(all_quality_pairs_rtv_audit)

shape: (52, 4)
┌───────────┬───────┬───────────────────────┬───────┐
│ metric_id ┆ count ┆ pair_type             ┆ param │
│ ---       ┆ ---   ┆ ---                   ┆ ---   │
│ str       ┆ u32   ┆ str                   ┆ str   │
╞═══════════╪═══════╪═══════════════════════╪═══════╡
│ metric_0  ┆ 0     ┆ value_without_quality ┆ RR    │
│ metric_1  ┆ 69    ┆ quality_without_value ┆ RR    │
│ metric_2  ┆ 0     ┆ value_without_quality ┆ TN    │
│ metric_3  ┆ 1711  ┆ quality_without_value ┆ TN    │
│ metric_4  ┆ 0     ┆ value_without_quality ┆ HTN   │
│ …         ┆ …     ┆ …                     ┆ …     │
│ metric_47 ┆ 17    ┆ quality_without_value ┆ DXI3S │
│ metric_48 ┆ 0     ┆ value_without_quality ┆ HXI3S │
│ metric_49 ┆ 17    ┆ quality_without_value ┆ HXI3S │
│ metric_50 ┆ 0     ┆ value_without_quality ┆ DRR   │
│ metric_51 ┆ 1644  ┆ quality_without_value ┆ DRR   │
└───────────┴───────┴───────────────────────┴───────┘


In [20]:
all_quality_pairs_rtv_matrix = (
    all_quality_pairs_rtv_audit
    .pivot(
        on="pair_type",
        index="param",
        values="count",
    )
)
with pl.Config(tbl_rows=30):
    print(all_quality_pairs_rtv_matrix)

shape: (26, 3)
┌────────┬───────────────────────┬───────────────────────┐
│ param  ┆ value_without_quality ┆ quality_without_value │
│ ---    ┆ ---                   ┆ ---                   │
│ str    ┆ u32                   ┆ u32                   │
╞════════╪═══════════════════════╪═══════════════════════╡
│ RR     ┆ 0                     ┆ 69                    │
│ TN     ┆ 0                     ┆ 1711                  │
│ HTN    ┆ 0                     ┆ 2066                  │
│ TX     ┆ 0                     ┆ 1705                  │
│ HTX    ┆ 0                     ┆ 2082                  │
│ TM     ┆ 0                     ┆ 1714                  │
│ TNTXM  ┆ 0                     ┆ 1                     │
│ TAMPLI ┆ 0                     ┆ 1                     │
│ TNSOL  ┆ 0                     ┆ 24                    │
│ TN50   ┆ 0                     ┆ 0                     │
│ DG     ┆ 0                     ┆ 1761                  │
│ FFM    ┆ 0                     ┆ 38    

##### 2.4.1.2. Orphan quality codes audit

###### 2.4.1.2.1 Orphan quality codes audit. Slow version (`concat`)

In [ ]:
audit_parts: list[pl.LazyFrame] = []

for param in dwl.RR_T_VENT_ALL_PARAMS:
    value_col = param.lower()
    quality_col = f"q{value_col}"

    orphan_quality_audit_rtv_all = audit_parts.append(
        lf_rtv_wide_all
        .filter(
            pl.col(value_col).is_null()
            & pl.col(quality_col).is_not_null()
        )
        .with_columns(
            quality_column=pl.lit(quality_col),
        )
        .group_by("quality_column", quality_col)
        .len()
        .sort(quality_col)
        .rename({quality_col: "quality_code_value"})
    )

orphan_quality_audit_rtv_all_concat = pl.concat(audit_parts).collect()

with pl.Config(tbl_rows=100):
    print(orphan_quality_audit_rtv_all_concat)

shape: (39, 3)
┌────────────────┬────────────────────┬──────┐
│ quality_column ┆ quality_code_value ┆ len  │
│ ---            ┆ ---                ┆ ---  │
│ str            ┆ u8                 ┆ u32  │
╞════════════════╪════════════════════╪══════╡
│ qrr            ┆ 1                  ┆ 69   │
│ qtn            ┆ 1                  ┆ 1683 │
│ qtn            ┆ 9                  ┆ 28   │
│ qhtn           ┆ 1                  ┆ 2032 │
│ qhtn           ┆ 9                  ┆ 34   │
│ qtx            ┆ 1                  ┆ 1687 │
│ qtx            ┆ 9                  ┆ 18   │
│ qhtx           ┆ 1                  ┆ 2051 │
│ qhtx           ┆ 9                  ┆ 31   │
│ qtm            ┆ 1                  ┆ 1711 │
│ qtm            ┆ 9                  ┆ 3    │
│ qtntxm         ┆ 1                  ┆ 1    │
│ qtampli        ┆ 1                  ┆ 1    │
│ qtnsol         ┆ 1                  ┆ 15   │
│ qtnsol         ┆ 9                  ┆ 9    │
│ qdg            ┆ 1                  ┆ 1751 

In [ ]:
quality_columns = pl.DataFrame({
    "quality_column": [f"q{param.lower()}" for param in dwl.RR_T_VENT_ALL_PARAMS],
})

quality_codes = pl.DataFrame({
    "quality_code_value": [0, 1, 2, 9],
})

quality_combinations = quality_columns.join(quality_codes, how="cross")

print(quality_combinations)

q_join = (
    quality_combinations
    .join(
        orphan_quality_audit_rtv_all_concat,
        how="left",
        on=["quality_column", "quality_code_value"],
    )
    .fill_null(0)
)

qm = q_join.pivot(
    on="quality_code_value",
    index="quality_column",
    values="len",
)

print(qm)

shape: (104, 2)
┌────────────────┬────────────────────┐
│ quality_column ┆ quality_code_value │
│ ---            ┆ ---                │
│ str            ┆ i64                │
╞════════════════╪════════════════════╡
│ qrr            ┆ 0                  │
│ qrr            ┆ 1                  │
│ qrr            ┆ 2                  │
│ qrr            ┆ 9                  │
│ qtn            ┆ 0                  │
│ …              ┆ …                  │
│ qhxi3s         ┆ 9                  │
│ qdrr           ┆ 0                  │
│ qdrr           ┆ 1                  │
│ qdrr           ┆ 2                  │
│ qdrr           ┆ 9                  │
└────────────────┴────────────────────┘
shape: (26, 5)
┌────────────────┬─────┬──────┬─────┬─────┐
│ quality_column ┆ 0   ┆ 1    ┆ 2   ┆ 9   │
│ ---            ┆ --- ┆ ---  ┆ --- ┆ --- │
│ str            ┆ u32 ┆ u32  ┆ u32 ┆ u32 │
╞════════════════╪═════╪══════╪═════╪═════╡
│ qrr            ┆ 0   ┆ 69   ┆ 0   ┆ 0   │
│ qtn            ┆ 0   ┆ 

###### 2.4.1.2.2 Orphan quality codes audit. Slow version (`concat`)

In [23]:
QUALITY_CODES = (0, 1, 2, 9)

audit_expressions = []
audit_metadata = []

for param in dwl.RR_T_VENT_ALL_PARAMS:
    param_col = param.lower()
    quality_col = f"q{param_col}"

    for quality_code in QUALITY_CODES:
        metric_id = f"metric_{len(audit_metadata)}"

        audit_expressions.append(
            (
                pl.col(param_col).is_null()
                & (pl.col(quality_col) == quality_code)
            )
            .sum()
            .alias(metric_id)
        )

        audit_metadata.append({
            "metric_id": metric_id,
            "quality_column": quality_col,
            "quality_code_value": quality_code,
        })

audit_counts = (
    lf_rtv_wide_all
    .select(audit_expressions)
    .collect()
)

In [ ]:
audit_metadata

[{'metric_id': 'metric_0', 'quality_column': 'qrr', 'quality_code_value': 0},
 {'metric_id': 'metric_1', 'quality_column': 'qrr', 'quality_code_value': 1},
 {'metric_id': 'metric_2', 'quality_column': 'qrr', 'quality_code_value': 2},
 {'metric_id': 'metric_3', 'quality_column': 'qrr', 'quality_code_value': 9},
 {'metric_id': 'metric_4', 'quality_column': 'qtn', 'quality_code_value': 0},
 {'metric_id': 'metric_5', 'quality_column': 'qtn', 'quality_code_value': 1},
 {'metric_id': 'metric_6', 'quality_column': 'qtn', 'quality_code_value': 2},
 {'metric_id': 'metric_7', 'quality_column': 'qtn', 'quality_code_value': 9},
 {'metric_id': 'metric_8', 'quality_column': 'qhtn', 'quality_code_value': 0},
 {'metric_id': 'metric_9', 'quality_column': 'qhtn', 'quality_code_value': 1},
 {'metric_id': 'metric_10', 'quality_column': 'qhtn', 'quality_code_value': 2},
 {'metric_id': 'metric_11', 'quality_column': 'qhtn', 'quality_code_value': 9},
 {'metric_id': 'metric_12', 'quality_column': 'qtx', 'qual

In [24]:
audit_counts_long = audit_counts.unpivot(
    variable_name="metric_id",
    value_name="len",
)
audit_counts_long

metric_id,len
str,u32
"""metric_0""",0
"""metric_1""",69
"""metric_2""",0
"""metric_3""",0
"""metric_4""",0
…,…
"""metric_99""",9
"""metric_100""",0
"""metric_101""",1419


In [25]:
audit_metadata_df = pl.DataFrame(
    audit_metadata,
    schema_overrides={
        "quality_code_value": pl.UInt8,
    },
)
audit_metadata_df

metric_id,quality_column,quality_code_value
str,str,u8
"""metric_0""","""qrr""",0
"""metric_1""","""qrr""",1
"""metric_2""","""qrr""",2
"""metric_3""","""qrr""",9
"""metric_4""","""qtn""",0
…,…,…
"""metric_99""","""qhxi3s""",9
"""metric_100""","""qdrr""",0
"""metric_101""","""qdrr""",1


In [26]:
orphan_quality_audit_rtv_all = (
    audit_counts_long
    .join(
        audit_metadata_df,
        how="left",
        on="metric_id",
        validate="1:1",
    )
)

orphan_quality_audit_rtv_all

metric_id,len,quality_column,quality_code_value
str,u32,str,u8
"""metric_0""",0,"""qrr""",0
"""metric_1""",69,"""qrr""",1
"""metric_2""",0,"""qrr""",2
"""metric_3""",0,"""qrr""",9
"""metric_4""",0,"""qtn""",0
…,…,…,…
"""metric_99""",9,"""qhxi3s""",9
"""metric_100""",0,"""qdrr""",0
"""metric_101""",1419,"""qdrr""",1


In [27]:
orphan_quality_matrix_rtv_all = orphan_quality_audit_rtv_all.pivot(
    on="quality_code_value",
    index="quality_column",
    values="len",
    sort_columns=True,
)
with pl.Config(tbl_rows=30):
    print(orphan_quality_matrix_rtv_all)

shape: (26, 5)
┌────────────────┬─────┬──────┬─────┬─────┐
│ quality_column ┆ 0   ┆ 1    ┆ 2   ┆ 9   │
│ ---            ┆ --- ┆ ---  ┆ --- ┆ --- │
│ str            ┆ u32 ┆ u32  ┆ u32 ┆ u32 │
╞════════════════╪═════╪══════╪═════╪═════╡
│ qrr            ┆ 0   ┆ 69   ┆ 0   ┆ 0   │
│ qtn            ┆ 0   ┆ 1683 ┆ 0   ┆ 28  │
│ qhtn           ┆ 0   ┆ 2032 ┆ 0   ┆ 34  │
│ qtx            ┆ 0   ┆ 1687 ┆ 0   ┆ 18  │
│ qhtx           ┆ 0   ┆ 2051 ┆ 0   ┆ 31  │
│ qtm            ┆ 0   ┆ 1711 ┆ 0   ┆ 3   │
│ qtntxm         ┆ 0   ┆ 1    ┆ 0   ┆ 0   │
│ qtampli        ┆ 0   ┆ 1    ┆ 0   ┆ 0   │
│ qtnsol         ┆ 0   ┆ 15   ┆ 0   ┆ 9   │
│ qtn50          ┆ 0   ┆ 0    ┆ 0   ┆ 0   │
│ qdg            ┆ 0   ┆ 1751 ┆ 0   ┆ 10  │
│ qffm           ┆ 0   ┆ 37   ┆ 0   ┆ 1   │
│ qff2m          ┆ 0   ┆ 0    ┆ 0   ┆ 0   │
│ qfxy           ┆ 0   ┆ 40   ┆ 0   ┆ 1   │
│ qdxy           ┆ 0   ┆ 41   ┆ 0   ┆ 1   │
│ qhxy           ┆ 0   ┆ 42   ┆ 0   ┆ 1   │
│ qfxi           ┆ 0   ┆ 37   ┆ 0   ┆ 1   │
│ qdxi           

#### 2.4.2. `autres-parametres`

##### 2.4.2.1. Audit of parameter / quality code pairs

In [32]:
all_quality_pairs_metadata, all_quality_pairs_expressions = dwl.quality_pair_audit_lf_2(lf=None, params=dwl.AUTRES_PARAMETRES_ALL_PARAMS)
all_quality_pairs_counts = (
    lf_ap_wide
    .select(all_quality_pairs_expressions)
    .collect()
)
print(all_quality_pairs_counts)

shape: (1, 82)
┌──────────┬──────────┬──────────┬──────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ metric_0 ┆ metric_1 ┆ metric_2 ┆ metric_3 ┆ … ┆ metric_78 ┆ metric_79 ┆ metric_80 ┆ metric_81 │
│ ---      ┆ ---      ┆ ---      ┆ ---      ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ u32      ┆ u32      ┆ u32      ┆ u32      ┆   ┆ u32       ┆ u32       ┆ u32       ┆ u32       │
╞══════════╪══════════╪══════════╪══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 0        ┆ 151      ┆ 0        ┆ 0        ┆ … ┆ 0         ┆ 0         ┆ 0         ┆ 0         │
└──────────┴──────────┴──────────┴──────────┴───┴───────────┴───────────┴───────────┴───────────┘


In [33]:
all_quality_pairs_counts_long = (
    all_quality_pairs_counts
    .unpivot(
        variable_name="metric_id",
        value_name="count",
    )
)
print(all_quality_pairs_counts_long)

shape: (82, 2)
┌───────────┬───────┐
│ metric_id ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ metric_0  ┆ 0     │
│ metric_1  ┆ 151   │
│ metric_2  ┆ 0     │
│ metric_3  ┆ 0     │
│ metric_4  ┆ 0     │
│ …         ┆ …     │
│ metric_77 ┆ 0     │
│ metric_78 ┆ 0     │
│ metric_79 ┆ 0     │
│ metric_80 ┆ 0     │
│ metric_81 ┆ 0     │
└───────────┴───────┘


In [34]:
all_quality_pairs_metadata_df = pl.DataFrame(all_quality_pairs_metadata)
print(all_quality_pairs_metadata_df)

shape: (82, 3)
┌───────────┬───────────────────────┬─────────┐
│ metric_id ┆ pair_type             ┆ param   │
│ ---       ┆ ---                   ┆ ---     │
│ str       ┆ str                   ┆ str     │
╞═══════════╪═══════════════════════╪═════════╡
│ metric_0  ┆ value_without_quality ┆ DHUMEC  │
│ metric_1  ┆ quality_without_value ┆ DHUMEC  │
│ metric_2  ┆ value_without_quality ┆ PMERM   │
│ metric_3  ┆ quality_without_value ┆ PMERM   │
│ metric_4  ┆ value_without_quality ┆ PMERMIN │
│ …         ┆ …                     ┆ …       │
│ metric_77 ┆ quality_without_value ┆ BA300   │
│ metric_78 ┆ value_without_quality ┆ TMERMIN │
│ metric_79 ┆ quality_without_value ┆ TMERMIN │
│ metric_80 ┆ value_without_quality ┆ TMERMAX │
│ metric_81 ┆ quality_without_value ┆ TMERMAX │
└───────────┴───────────────────────┴─────────┘


In [35]:
all_quality_pairs_ap_audit = (
    all_quality_pairs_counts_long
    .join(
        all_quality_pairs_metadata_df,
        on="metric_id",
        how="left",
        validate="1:1",
    )
)
print(all_quality_pairs_ap_audit)

shape: (82, 4)
┌───────────┬───────┬───────────────────────┬─────────┐
│ metric_id ┆ count ┆ pair_type             ┆ param   │
│ ---       ┆ ---   ┆ ---                   ┆ ---     │
│ str       ┆ u32   ┆ str                   ┆ str     │
╞═══════════╪═══════╪═══════════════════════╪═════════╡
│ metric_0  ┆ 0     ┆ value_without_quality ┆ DHUMEC  │
│ metric_1  ┆ 151   ┆ quality_without_value ┆ DHUMEC  │
│ metric_2  ┆ 0     ┆ value_without_quality ┆ PMERM   │
│ metric_3  ┆ 0     ┆ quality_without_value ┆ PMERM   │
│ metric_4  ┆ 0     ┆ value_without_quality ┆ PMERMIN │
│ …         ┆ …     ┆ …                     ┆ …       │
│ metric_77 ┆ 0     ┆ quality_without_value ┆ BA300   │
│ metric_78 ┆ 0     ┆ value_without_quality ┆ TMERMIN │
│ metric_79 ┆ 0     ┆ quality_without_value ┆ TMERMIN │
│ metric_80 ┆ 0     ┆ value_without_quality ┆ TMERMAX │
│ metric_81 ┆ 0     ┆ quality_without_value ┆ TMERMAX │
└───────────┴───────┴───────────────────────┴─────────┘


In [37]:
all_quality_pairs_ap_matrix = (
    all_quality_pairs_ap_audit
    .pivot(
        on="pair_type",
        index="param",
        values="count",
    )
)
with pl.Config(tbl_rows=50):
    print(all_quality_pairs_ap_matrix)

shape: (41, 3)
┌─────────────┬───────────────────────┬───────────────────────┐
│ param       ┆ value_without_quality ┆ quality_without_value │
│ ---         ┆ ---                   ┆ ---                   │
│ str         ┆ u32                   ┆ u32                   │
╞═════════════╪═══════════════════════╪═══════════════════════╡
│ DHUMEC      ┆ 0                     ┆ 151                   │
│ PMERM       ┆ 0                     ┆ 0                     │
│ PMERMIN     ┆ 0                     ┆ 2                     │
│ INST        ┆ 0                     ┆ 0                     │
│ GLOT        ┆ 0                     ┆ 5                     │
│ DIFT        ┆ 0                     ┆ 0                     │
│ DIRT        ┆ 0                     ┆ 0                     │
│ INFRART     ┆ 0                     ┆ 0                     │
│ UV          ┆ 0                     ┆ 0                     │
│ UV_INDICEX  ┆ 0                     ┆ 0                     │
│ SIGMA       ┆ 0        

##### 2.4.2.2. Orphan quality codes audit

In [31]:
QUALITY_CODES = (0, 1, 2, 9)

audit_expressions = []
audit_metadata = []

for param in dwl.AUTRES_PARAMETRES_ALL_PARAMS:
    param_col = param.lower()
    quality_col = f"q{param_col}"

    for quality_code in QUALITY_CODES:
        metric_id = f"metric_{len(audit_metadata)}"

        audit_expressions.append(
            (
                pl.col(param_col).is_null()
                & (pl.col(quality_col) == quality_code)
            )
            .sum()
            .alias(metric_id)
        )

        audit_metadata.append({
            "metric_id": metric_id,
            "quality_column": quality_col,
            "quality_code_value": quality_code,
        })

audit_counts = (
    lf_ap_wide
    .select(audit_expressions)
    .collect()
)

audit_counts_long = (
    audit_counts
    .unpivot(
        variable_name="metric_id",
        value_name="count",
    )
)

audit_metadata_df = pl.DataFrame(
    audit_metadata,
    schema_overrides={
        "quality_code_value": pl.UInt8,
    }
)

orphan_quality_audit_ap = (
    audit_counts_long
    .join(
        audit_metadata_df,
        on="metric_id",
        how="left",
        validate="1:1",
    )
)

orphan_quality_matrix_ap = (
    orphan_quality_audit_ap
    .pivot(
        on="quality_code_value",
        index="quality_column",
        values="count",
    )
)

with pl.Config(tbl_rows=50):
    print(orphan_quality_matrix_ap)

shape: (41, 5)
┌────────────────┬─────┬───────┬─────┬─────┐
│ quality_column ┆ 0   ┆ 1     ┆ 2   ┆ 9   │
│ ---            ┆ --- ┆ ---   ┆ --- ┆ --- │
│ str            ┆ u32 ┆ u32   ┆ u32 ┆ u32 │
╞════════════════╪═════╪═══════╪═════╪═════╡
│ qdhumec        ┆ 0   ┆ 151   ┆ 0   ┆ 0   │
│ qpmerm         ┆ 0   ┆ 0     ┆ 0   ┆ 0   │
│ qpmermin       ┆ 0   ┆ 1     ┆ 0   ┆ 1   │
│ qinst          ┆ 0   ┆ 0     ┆ 0   ┆ 0   │
│ qglot          ┆ 0   ┆ 5     ┆ 0   ┆ 0   │
│ qdift          ┆ 0   ┆ 0     ┆ 0   ┆ 0   │
│ qdirt          ┆ 0   ┆ 0     ┆ 0   ┆ 0   │
│ qinfrart       ┆ 0   ┆ 0     ┆ 0   ┆ 0   │
│ quv            ┆ 0   ┆ 0     ┆ 0   ┆ 0   │
│ quv_indicex    ┆ 0   ┆ 0     ┆ 0   ┆ 0   │
│ qsigma         ┆ 0   ┆ 0     ┆ 0   ┆ 0   │
│ qun            ┆ 0   ┆ 44    ┆ 0   ┆ 1   │
│ qhun           ┆ 0   ┆ 82    ┆ 0   ┆ 3   │
│ qux            ┆ 0   ┆ 27    ┆ 0   ┆ 1   │
│ qhux           ┆ 0   ┆ 45    ┆ 0   ┆ 1   │
│ qum            ┆ 0   ┆ 28    ┆ 0   ┆ 2   │
│ qdhumi40       ┆ 0   ┆ 54    ┆ 0   ┆ 2

##### 2.4.2.3. Drill-down on orphan quality codes for `ETPMON` and `NEIGETOTX`

In [40]:
with pl.Config(tbl_rows=20):
    print(dwl.drill_down_orphan_quality_pairs(lf_ap_wide, param="ETPMON", quality_code=1))

shape: (11, 5)
┌────────────┬──────────────────────┬───────┬────────────┬────────────┐
│ station_id ┆ station_name         ┆ count ┆ min_date   ┆ max_date   │
│ ---        ┆ ---                  ┆ ---   ┆ ---        ┆ ---        │
│ str        ┆ str                  ┆ u32   ┆ date       ┆ date       │
╞════════════╪══════════════════════╪═══════╪════════════╪════════════╡
│ 79049004   ┆ BRESSUIRE            ┆ 8774  ┆ 1990-11-10 ┆ 2026-04-07 │
│ 79134001   ┆ GLENAY_SAPC          ┆ 1238  ┆ 1993-02-01 ┆ 1996-07-31 │
│ 79174002   ┆ MELLE                ┆ 5774  ┆ 1990-02-18 ┆ 2026-04-05 │
│ 79191005   ┆ NIORT                ┆ 38    ┆ 1991-12-28 ┆ 2018-12-14 │
│ 79202003   ┆ PARTHENAY_SAPC       ┆ 1035  ┆ 1993-09-23 ┆ 1996-07-31 │
│ 79215001   ┆ POUGNE-HERISSON_SAPC ┆ 1267  ┆ 1992-10-02 ┆ 1996-07-31 │
│ 79309001   ┆ SCILLE               ┆ 7299  ┆ 2006-02-10 ┆ 2026-04-07 │
│ 79326004   ┆ THENEZAY STNA        ┆ 8699  ┆ 2002-05-29 ┆ 2026-04-07 │
│ 79329001   ┆ THOUARS-STNA         ┆ 3386  ┆ 199

In [ ]:
with pl.Config(tbl_rows=20):
    print(dwl.drill_down_orphan_quality_pairs(lf_ap_wide, param="NEIGETOTX", quality_code=1))

shape: (10, 5)
┌────────────┬──────────────────────┬───────┬────────────┬────────────┐
│ station_id ┆ station_name         ┆ count ┆ min_date   ┆ max_date   │
│ ---        ┆ ---                  ┆ ---   ┆ ---        ┆ ---        │
│ str        ┆ str                  ┆ u32   ┆ date       ┆ date       │
╞════════════╪══════════════════════╪═══════╪════════════╪════════════╡
│ 79049004   ┆ BRESSUIRE            ┆ 842   ┆ 1991-04-24 ┆ 1997-11-25 │
│ 79134001   ┆ GLENAY_SAPC          ┆ 835   ┆ 1994-05-30 ┆ 1999-11-21 │
│ 79174002   ┆ MELLE                ┆ 865   ┆ 1990-05-15 ┆ 1999-12-16 │
│ 79191005   ┆ NIORT                ┆ 2609  ┆ 1986-01-01 ┆ 1999-12-31 │
│ 79202003   ┆ PARTHENAY_SAPC       ┆ 978   ┆ 1993-10-01 ┆ 1999-11-07 │
│ 79215001   ┆ POUGNE-HERISSON_SAPC ┆ 1263  ┆ 1992-07-01 ┆ 1999-02-13 │
│ 79253002   ┆ LA TOUCHE STNA       ┆ 268   ┆ 1996-10-08 ┆ 1999-10-06 │
│ 79329001   ┆ THOUARS-STNA         ┆ 1008  ┆ 1993-09-24 ┆ 1998-07-29 │
│ 79342002   ┆ VERNOUX STNA         ┆ 772   ┆ 199

### 2.5 Audit of lines without measurements

In [43]:
print(
    lf_rtv_wide_all
    .filter(
        pl.col("all_measure_count") == 0
    )
    .select(
        "station_id",
        "station_name",
        "obs_date"
    )
    .group_by(
        "station_id",
        "station_name",
    )
    .agg(
        count=pl.len(),
        min_date=pl.col("obs_date").min(),
        max_date=pl.col("obs_date").max(),
    )
    .sort("station_id")
    .collect()
)

shape: (2, 5)
┌────────────┬───────────────────┬───────┬────────────┬────────────┐
│ station_id ┆ station_name      ┆ count ┆ min_date   ┆ max_date   │
│ ---        ┆ ---               ┆ ---   ┆ ---        ┆ ---        │
│ str        ┆ str               ┆ u32   ┆ date       ┆ date       │
╞════════════╪═══════════════════╪═══════╪════════════╪════════════╡
│ 79017001   ┆ NUEIL-LES-AUBIERS ┆ 396   ┆ 1962-09-01 ┆ 1970-12-31 │
│ 79176001   ┆ MENIGOUTE_SAPC    ┆ 31    ┆ 1964-01-01 ┆ 1964-01-31 │
└────────────┴───────────────────┴───────┴────────────┴────────────┘


In [45]:
with pl.Config(tbl_rows=30):
    print(
        lf_ap_wide
        .filter(
            pl.col("all_measure_count") == 0
        )
        .select(
            "station_id",
            "station_name",
            "obs_date"
        )
        .group_by(
            "station_id",
            "station_name",
        )
        .agg(
            count=pl.len(),
            min_date=pl.col("obs_date").min(),
            max_date=pl.col("obs_date").max(),
        )
        .sort("station_id")
        .collect()
    )

shape: (28, 5)
┌────────────┬───────────────────────────┬───────┬────────────┬────────────┐
│ station_id ┆ station_name              ┆ count ┆ min_date   ┆ max_date   │
│ ---        ┆ ---                       ┆ ---   ┆ ---        ┆ ---        │
│ str        ┆ str                       ┆ u32   ┆ date       ┆ date       │
╞════════════╪═══════════════════════════╪═══════╪════════════╪════════════╡
│ 79013002   ┆ SANZAY                    ┆ 1     ┆ 1984-01-20 ┆ 1984-01-20 │
│ 79043001   ┆ BOUILLE-LORETZ            ┆ 1     ┆ 2000-08-05 ┆ 2000-08-05 │
│ 79047001   ┆ BOUSSAIS ECOLE            ┆ 2     ┆ 1951-05-06 ┆ 1951-05-28 │
│ 79049001   ┆ BRESSUIRE                 ┆ 3     ┆ 1984-01-19 ┆ 2002-08-13 │
│ 79061001   ┆ CELLES-SUR-BELL           ┆ 2     ┆ 1960-01-19 ┆ 1967-11-02 │
│ 79079001   ┆ MAULEON                   ┆ 5     ┆ 1956-01-23 ┆ 1984-01-26 │
│ 79103001   ┆ COURLAY PLACE DE L'EGLISE ┆ 10    ┆ 1957-11-03 ┆ 1963-02-18 │
│ 79134001   ┆ GLENAY_SAPC               ┆ 3     ┆ 1952-11-27

### 2.6 Audit of parameters with uncertain types

In [48]:
print(dwl.audit_numeric_types_lf(lf_ap_wide, dwl.FIELDS_TO_AUDIT))

shape: (8, 5)
┌────────────┬──────┬───────┬──────────┬──────────────────┐
│ column     ┆ min  ┆ max   ┆ n_unique ┆ fractional_count │
│ ---        ┆ ---  ┆ ---   ┆ ---      ┆ ---              │
│ str        ┆ f64  ┆ f64   ┆ i64      ┆ i64              │
╞════════════╪══════╪═══════╪══════════╪══════════════════╡
│ hneigef    ┆ 0.0  ┆ 18.0  ┆ 16       ┆ 0                │
│ neigetotx  ┆ 0.0  ┆ 18.0  ┆ 17       ┆ 0                │
│ neigetot06 ┆ 0.0  ┆ 18.0  ┆ 15       ┆ 0                │
│ uv_indicex ┆ null ┆ null  ┆ 0        ┆ 0                │
│ sigma      ┆ 0.0  ┆ 101.0 ┆ 102      ┆ 0                │
│ un         ┆ 1.0  ┆ 100.0 ┆ 100      ┆ 0                │
│ ux         ┆ 34.0 ┆ 100.0 ┆ 62       ┆ 0                │
│ um         ┆ 20.0 ┆ 100.0 ┆ 78       ┆ 0                │
└────────────┴──────┴───────┴──────────┴──────────────────┘


## 3. Wide -> long transformation

Pour RTV, deux colonnes status_fxi3s et status_dxi3s sont des attributs des paramètres fxi3s et dxi3s (codes 0/1). On prévoit donc une colonne status_code dans la représentation long des données RTV pouvant valoir, pour les paramètres concernés, 0 ou 1 (ou null), ou null sinon.

Target grain?

### `RR-T-Vent`

In [58]:
value_columns_rtv = [
    param.lower()
    for param in dwl.RR_T_VENT_ALL_PARAMS
]

quality_columns_rtv = [
    f"q{param.lower()}"
    for param in dwl.RR_T_VENT_ALL_PARAMS
]

lf_long_rtv_all_values = (
    lf_rtv_wide_all
    .unpivot(
        on=value_columns_rtv,
        index=["station_id", "obs_date", "source_file"],
        variable_name="param_code",
        value_name="value",
    )
    .filter(pl.col("value").is_not_null())
)

lf_long_rtv_all_quality = (
    lf_rtv_wide_all
    .unpivot(
        on=quality_columns_rtv,
        index=["station_id", "obs_date", "source_file"],
        variable_name="quality_param_code",
        value_name="quality_code",
    )
    .filter(pl.col("quality_code").is_not_null())
    .with_columns(
        pl.col("quality_param_code")
        .str.slice(1)
        .alias("param_code")
    )
    .drop("quality_param_code")
)

join_keys = ["source_file", "station_id", "obs_date", "param_code"]

lf_long_rtv_all = (
    lf_long_rtv_all_values
    .join(
        lf_long_rtv_all_quality,
        on=join_keys,
        how="left",
        validate="1:1",
    )
)

lf_rtv_status = (
    lf_rtv_wide_all
    .select(
        "station_id",
        "obs_date",
        "status_fxi3s",
        "status_dxi3s",
        "source_file",
    )
)

lf_long_rtv_bench = (
    lf_long_rtv_all
    .join(
        lf_rtv_status,
        on=["source_file", "station_id", "obs_date"],
        how="left",
        validate="m:1",
    )
    .with_columns(
        pl.when(pl.col("param_code") == "fxi3s")
        .then(pl.col("status_fxi3s"))
        .when(pl.col("param_code") == "dxi3s")
        .then(pl.col("status_dxi3s"))
        .otherwise(None)
        .cast(pl.UInt8)
        .alias("status_code")
    )
    .drop(
        "status_fxi3s",
        "status_dxi3s",
    )
)

# df_long_rtv_bench = lf_long_rtv_all.collect()

# print("""df_long_rtv_all.filter((pl.col("status_code").is_not_null()) & (pl.col("value").is_null()))""")
# print(df_long_rtv_all.filter((pl.col("status_code").is_not_null()) & (pl.col("value").is_null())))
# print("""df_long_rtv_all.filter((pl.col("status_code").is_null()) & (pl.col("value").is_not_null()))""")
# print(df_long_rtv_all.filter((pl.col("status_code").is_null()) & (pl.col("value").is_not_null())))
# print("""df_long_rtv_all.filter((pl.col("status_code").is_not_null()) & ~((pl.col("param_code") == "fxi3s") | (pl.col("param_code") == "dxi3s")))""")
# print(df_long_rtv_all.filter((pl.col("status_code").is_not_null()) & ~((pl.col("param_code") == "fxi3s") | (pl.col("param_code") == "dxi3s"))))
# print("""df_long_rtv_all.filter((pl.col("status_code").is_not_null()))""")
# print(df_long_rtv_all.filter((pl.col("status_code").is_not_null())))


# print("df_long_rtv_all")
# print(df_long_rtv_all)

# print("""df_rtv_all.estimated_size("mb")""")
# print(df_rtv_all.estimated_size("mb"))
# print("""df_long_rtv_all.estimated_size("mb")""")
# print(df_long_rtv_all.estimated_size("mb"))


In [61]:
df_rtv_wide_bench = (
    lf_rtv_wide_all
    .drop(["station_name", "source_file", "all_measure_count", "measure_signature"])
    .collect()
)

In [52]:
# print("""df_long_rtv_all.filter((pl.col("status_code").is_not_null()) & (pl.col("value").is_null()))""")
# print(df_long_rtv_all.filter((pl.col("status_code").is_not_null()) & (pl.col("value").is_null())))
# print("""df_long_rtv_all.filter((pl.col("status_code").is_null()) & (pl.col("value").is_not_null()))""")
# print(df_long_rtv_all.filter((pl.col("status_code").is_null()) & (pl.col("value").is_not_null())))
# print("""df_long_rtv_all.filter((pl.col("status_code").is_not_null()) & ~((pl.col("param_code") == "fxi3s") | (pl.col("param_code") == "dxi3s")))""")
# print(df_long_rtv_all.filter((pl.col("status_code").is_not_null()) & ~((pl.col("param_code") == "fxi3s") | (pl.col("param_code") == "dxi3s"))))
# print("""df_long_rtv_all.filter((pl.col("status_code").is_not_null()))""")
# print(df_long_rtv_all.filter((pl.col("status_code").is_not_null())))


# print("df_long_rtv_all")
# print(df_long_rtv_all)

print("""df_rtv_all.estimated_size("mb")""")
print(df_rtv_wide_all.estimated_size("mb"))
print("""df_long_rtv_all.estimated_size("mb")""")
print(df_long_rtv_all.estimated_size("mb"))

df_rtv_all.estimated_size("mb")
276.61173915863037
df_long_rtv_all.estimated_size("mb")
248.50394916534424


### `autres-parametres`

In [ ]:
value_columns_ap = [
    param.lower()
    for param in dwl.AUTRES_PARAMETRES_ALL_PARAMS
]

quality_columns_ap = [
    f"q{param.lower()}"
    for param in dwl.AUTRES_PARAMETRES_ALL_PARAMS
]

lf_long_ap_values = (
    lf_ap_wide
    .unpivot(
        on=value_columns_ap,
        index=["station_id", "obs_date", "source_file"],
        variable_name="param_code",
        value_name="value",
    )
    .filter(pl.col("value").is_not_null())
)

lf_long_ap_quality = (
    lf_ap_wide
    .unpivot(
        on=quality_columns_ap,
        index=["station_id", "obs_date", "source_file"],
        variable_name="quality_param_code",
        value_name="quality_code",
    )
    .filter(pl.col("quality_code").is_not_null())
    .with_columns(
        pl.col("quality_param_code")
        .str.slice(1)
        .alias("param_code")
    )
    .drop("quality_param_code")
)

join_keys = ["source_file", "station_id", "obs_date", "param_code"]

lf_long_ap = (
    lf_long_ap_values
    .join(
        lf_long_ap_quality,
        on=join_keys,
        how="left",
        validate="1:1",
    )
)


lf_long_ap_bench = (
    lf_long_ap
    .with_columns(
        status_code=pl.lit(None, dtype=pl.UInt8),
    )
)

# with pl.Config(tbl_rows=100):
#     print(lf_long_ap_bench.collect())

# df_long_rtv_bench = lf_long_rtv_all.collect()

# print("""df_long_rtv_all.filter((pl.col("status_code").is_not_null()) & (pl.col("value").is_null()))""")
# print(df_long_rtv_all.filter((pl.col("status_code").is_not_null()) & (pl.col("value").is_null())))
# print("""df_long_rtv_all.filter((pl.col("status_code").is_null()) & (pl.col("value").is_not_null()))""")
# print(df_long_rtv_all.filter((pl.col("status_code").is_null()) & (pl.col("value").is_not_null())))
# print("""df_long_rtv_all.filter((pl.col("status_code").is_not_null()) & ~((pl.col("param_code") == "fxi3s") | (pl.col("param_code") == "dxi3s")))""")
# print(df_long_rtv_all.filter((pl.col("status_code").is_not_null()) & ~((pl.col("param_code") == "fxi3s") | (pl.col("param_code") == "dxi3s"))))
# print("""df_long_rtv_all.filter((pl.col("status_code").is_not_null()))""")
# print(df_long_rtv_all.filter((pl.col("status_code").is_not_null())))


# print("df_long_rtv_all")
# print(df_long_rtv_all)

# print("""df_rtv_all.estimated_size("mb")""")
# print(df_rtv_all.estimated_size("mb"))
# print("""df_long_rtv_all.estimated_size("mb")""")
# print(df_long_rtv_all.estimated_size("mb"))


shape: (2_148_378, 7)
┌────────────┬────────────┬──────────────────────┬────────────┬───────┬──────────────┬─────────────┐
│ station_id ┆ obs_date   ┆ source_file          ┆ param_code ┆ value ┆ quality_code ┆ status_code │
│ ---        ┆ ---        ┆ ---                  ┆ ---        ┆ ---   ┆ ---          ┆ ---         │
│ str        ┆ date       ┆ str                  ┆ str        ┆ f64   ┆ u8           ┆ u8          │
╞════════════╪════════════╪══════════════════════╪════════════╪═══════╪══════════════╪═════════════╡
│ 79134001   ┆ 2004-10-18 ┆ Q_79_previous-1950-2 ┆ dhumec     ┆ 0.0   ┆ 9            ┆ null        │
│            ┆            ┆ 024_autres…          ┆            ┆       ┆              ┆             │
│ 79134001   ┆ 2004-10-19 ┆ Q_79_previous-1950-2 ┆ dhumec     ┆ 0.0   ┆ 9            ┆ null        │
│            ┆            ┆ 024_autres…          ┆            ┆       ┆              ┆             │
│ 79134001   ┆ 2004-10-20 ┆ Q_79_previous-1950-2 ┆ dhumec     ┆ 0.0  

In [56]:
df_ap_wide_bench = (
    lf_ap_wide
    .drop(["station_name", "source_file", "all_measure_count", "measure_signature"])
    .collect()
)

### Comparisons

In [59]:
lf_daily_long = pl.concat(
    [lf_long_rtv_bench, lf_long_ap_bench],
    how="vertical",
)

In [ ]:
df_daily_long = (
    lf_daily_long
    .collect()
)

In [62]:
wide_long_comparison = pl.DataFrame({
    "name": [
        "df_rtv_wide_bench",
        "df_ap_wide_bench",
        "wide_rtv_ap_sum",
        "daily_long",
    ],
    "height": [
        df_rtv_wide_bench.height,
        df_ap_wide_bench.height,
        df_rtv_wide_bench.height + df_ap_wide_bench.height,
        df_daily_long.height,
    ],
    "width": [
        df_rtv_wide_bench.width,
        df_ap_wide_bench.width,
        df_rtv_wide_bench.width + df_ap_wide_bench.width,
        df_daily_long.width,
    ],
    "estimated_size": [
        df_rtv_wide_bench.estimated_size("mb"),
        df_ap_wide_bench.estimated_size("mb"),
        df_rtv_wide_bench.estimated_size("mb") + df_ap_wide_bench.estimated_size("mb"),
        df_daily_long.estimated_size("mb"),
    ],
})

print(wide_long_comparison)

NameError: name 'df_daily_long' is not defined

## REARRANGE

With the scope limited to the department 79, the resulting DataFrame contains 1,172,316 lines.

### Core RR-T-Vent parameters (5 parameters)

In [ ]:
lf_rtv_wide_core = dwl.add_measure_count_lf(lf_rtv_wide_core, dwl.RR_T_VENT_CORE_PARAMS, "core_measure_count")
expected_long_rows_rtv_core = (
    lf_rtv_wide_core
    .select(pl.col("core_measure_count").sum())
    .collect()
    .item()
)
mean_measures_per_wide_row_rtv_core = (
    expected_long_rows_rtv_core / df_rtv_wide_core.height
)
measure_density_rtv_core = (
    expected_long_rows_rtv_core
    / (df_rtv_wide_core.height * len(dwl.RR_T_VENT_CORE_PARAMS))
)

with pl.Config(tbl_cols=20, tbl_rows=100):
    print("RR-T-Vent core parameters:")
    print(f"height: {df_rtv_wide_core.height}")
    print(f"expected_long_core_rows: {expected_long_rows_rtv_core}")
    print(f"mean_measures_per_wide_row: {mean_measures_per_wide_row_rtv_core}")
    print(f"measure_density: {measure_density_rtv_core}")

    print(
        lf_rtv_wide_core
        .group_by("core_measure_count")
        .agg(
            count=pl.len(),
            pct=pl.len() / df_rtv_wide_core.height * 100
        )
        .sort("core_measure_count")
        .collect()
    )

height: 1172316
expected_long_core_rows: 2637628
mean_measures_per_wide_row: 2.249929199976798
measure_density: 0.4499858399953596
shape: (6, 3)
┌────────────────────┬────────┬───────────┐
│ core_measure_count ┆ count  ┆ pct       │
│ ---                ┆ ---    ┆ ---       │
│ u32                ┆ u32    ┆ f64       │
╞════════════════════╪════════╪═══════════╡
│ 0                  ┆ 442    ┆ 0.037703  │
│ 1                  ┆ 733868 ┆ 62.599845 │
│ 2                  ┆ 28     ┆ 0.002388  │
│ 3                  ┆ 941    ┆ 0.080268  │
│ 4                  ┆ 284304 ┆ 24.251482 │
│ 5                  ┆ 152733 ┆ 13.028313 │
└────────────────────┴────────┴───────────┘


In [ ]:
lf_rtv_wide_core = lf_rtv_wide_core.with_columns(
    dwl.measure_signature_expr(dwl.RR_T_VENT_CORE_PARAMS)
    .alias("measure_signature")
)

patterns_rtv_core = (
    lf_rtv_wide_core
    .group_by(
        "core_measure_count",
        "measure_signature",
    )
    .agg(
        count=pl.len(),
    )
    .with_columns(
        pct=pl.col("count") / df_rtv_wide_core.height * 100
    )
    .sort("count", descending=True)
    .collect()
)

with pl.Config(tbl_cols=20, tbl_rows=100, fmt_str_lengths=300):   
    print(patterns_rtv_core.head(30))

shape: (14, 4)
┌────────────────────┬───────────────────┬────────┬───────────┐
│ core_measure_count ┆ measure_signature ┆ count  ┆ pct       │
│ ---                ┆ ---               ┆ ---    ┆ ---       │
│ u32                ┆ str               ┆ u32    ┆ f64       │
╞════════════════════╪═══════════════════╪════════╪═══════════╡
│ 1                  ┆ RR                ┆ 733846 ┆ 62.597968 │
│ 4                  ┆ RR,TN,TX,TNTXM    ┆ 282433 ┆ 24.091883 │
│ 5                  ┆ RR,TN,TX,TM,TNTXM ┆ 152733 ┆ 13.028313 │
│ 4                  ┆ TN,TX,TM,TNTXM    ┆ 1871   ┆ 0.159599  │
│ 3                  ┆ TN,TX,TNTXM       ┆ 934    ┆ 0.079671  │
│ 0                  ┆                   ┆ 442    ┆ 0.037703  │
│ 2                  ┆ RR,TN             ┆ 16     ┆ 0.001365  │
│ 1                  ┆ TX                ┆ 14     ┆ 0.001194  │
│ 2                  ┆ RR,TX             ┆ 9      ┆ 0.000768  │
│ 3                  ┆ RR,TN,TM          ┆ 7      ┆ 0.000597  │
│ 1                  ┆ TM

#### Value / quality consistency

The CSV source data files contain daily observations of meteorological parameters. Each parameter corresponds to a column in the files.
Each parameter value has a corresponding quality code. In the data files, quality codes are listed in a separate column. For a given parameter, the quality code is in the column immediately to the right of the column containing the parameter itself.

In [ ]:
rtv_core_quality_audit = dwl.quality_pair_audit_lf(lf_rtv_wide_core, dwl.RR_T_VENT_CORE_PARAMS)

print(rtv_core_quality_audit)

shape: (10, 2)
┌─────────────────────────────┬───────┐
│ line_type                   ┆ count │
│ ---                         ┆ ---   │
│ str                         ┆ u32   │
╞═════════════════════════════╪═══════╡
│ RR_value_without_quality    ┆ 0     │
│ RR_quality_without_value    ┆ 69    │
│ TN_value_without_quality    ┆ 0     │
│ TN_quality_without_value    ┆ 1711  │
│ TX_value_without_quality    ┆ 0     │
│ TX_quality_without_value    ┆ 1705  │
│ TM_value_without_quality    ┆ 0     │
│ TM_quality_without_value    ┆ 1714  │
│ TNTXM_value_without_quality ┆ 0     │
│ TNTXM_quality_without_value ┆ 1     │
└─────────────────────────────┴───────┘


In [ ]:
orphan_quality_audit_rtv_core = pl.DataFrame()
for param in dwl.RR_T_VENT_CORE_PARAMS:
    value_col = param.lower()
    quality_col = f"q{value_col}"

    orphan_quality_audit_rtv_core = pl.concat([
        orphan_quality_audit_rtv_core,
        lf_rtv_wide_core
        .filter(
            pl.col(value_col).is_null()
            & pl.col(quality_col).is_not_null()
        )
        .with_columns(
            quality_column=pl.lit(quality_col),
        )
        .group_by("quality_column", quality_col)
        .len()
        .sort(quality_col)
        .rename({quality_col: "quality_code_value"})
        .collect()
    ])

with pl.Config(tbl_rows=100):
    print(orphan_quality_audit_rtv_core)

shape: (8, 3)
┌────────────────┬────────────────────┬──────┐
│ quality_column ┆ quality_code_value ┆ len  │
│ ---            ┆ ---                ┆ ---  │
│ str            ┆ u8                 ┆ u32  │
╞════════════════╪════════════════════╪══════╡
│ qrr            ┆ 1                  ┆ 69   │
│ qtn            ┆ 1                  ┆ 1683 │
│ qtn            ┆ 9                  ┆ 28   │
│ qtx            ┆ 1                  ┆ 1687 │
│ qtx            ┆ 9                  ┆ 18   │
│ qtm            ┆ 1                  ┆ 1711 │
│ qtm            ┆ 9                  ┆ 3    │
│ qtntxm         ┆ 1                  ┆ 1    │
└────────────────┴────────────────────┴──────┘


### All RR-T-Vent parameters (26 parameters)

#### Measurement density

In [ ]:
lf_rtv_wide_all = dwl.add_measure_count_lf(lf_rtv_wide_all, dwl.RR_T_VENT_ALL_PARAMS, "all_measure_count")

summary = (
    lf_rtv_wide_all
    .select(
        n_rows=pl.len(),
        n_measures=pl.col("all_measure_count").sum(),
    )
    .with_columns(
        mean_measures_per_wide_row=pl.col("n_measures") / pl.col("n_rows"),
        measure_density=pl.col("n_measures") / (pl.col("n_rows") * len(dwl.RR_T_VENT_ALL_PARAMS)),
    )
    .collect()
)

print("RR-T-Vent all parameters:")
print(summary)

with pl.Config(tbl_cols=20, tbl_rows=100):
    print(
        lf_rtv_wide_all
        .group_by("all_measure_count")
        .len(name="count")
        .with_columns(
            pct=pl.col("count") / pl.col("count").sum() * 100,
        )
        .sort("all_measure_count")
        .collect()
    )

shape: (1, 4)
┌─────────┬────────────┬────────────────────────────┬─────────────────┐
│ n_rows  ┆ n_measures ┆ mean_measures_per_wide_row ┆ measure_density │
│ ---     ┆ ---        ┆ ---                        ┆ ---             │
│ u32     ┆ u32        ┆ f64                        ┆ f64             │
╞═════════╪════════════╪════════════════════════════╪═════════════════╡
│ 1172316 ┆ 4242670    ┆ 3.61905                    ┆ 0.139194        │
└─────────┴────────────┴────────────────────────────┴─────────────────┘
RR-T-Vent all parameters:
shape: (23, 3)
┌───────────────────┬────────┬───────────┐
│ all_measure_count ┆ count  ┆ pct       │
│ ---               ┆ ---    ┆ ---       │
│ u32               ┆ u32    ┆ f64       │
╞═══════════════════╪════════╪═══════════╡
│ 0                 ┆ 427    ┆ 0.036424  │
│ 1                 ┆ 733810 ┆ 62.594898 │
│ 2                 ┆ 69     ┆ 0.005886  │
│ 3                 ┆ 4      ┆ 0.000341  │
│ 4                 ┆ 861    ┆ 0.073444  │
│ 5        

#### Recurring parameter signatures

In [ ]:
lf_rtv_wide_all = lf_rtv_wide_all.with_columns(
    dwl.measure_signature_expr(dwl.RR_T_VENT_ALL_PARAMS)
    .alias("measure_signature")
)

patterns_rtv_all = (
    lf_rtv_wide_all
    .group_by(
        "all_measure_count",
        "measure_signature",
    )
    .agg(
        count=pl.len(),
    )
    .with_columns(
        pct=pl.col("count") / df_rtv_wide_all.height * 100
    )
    .sort("count", descending=True)
    .collect()
)

patterns_rtv_all_head = patterns_rtv_all.head(30)
print(f"patterns_rtv_all.shape: {patterns_rtv_all.shape}")
print(f"patterns_rtv_all sum pct: {patterns_rtv_all.select("pct").sum().item()}")
with pl.Config(tbl_cols=20, tbl_rows=100, fmt_str_lengths=300):   
    print(patterns_rtv_all_head)

patterns_rtv_all.shape: (344, 4)
patterns_rtv_all sum pct: 100.00000000000006
shape: (30, 4)
┌───────────────────┬─────────────────────────────────────────────────────────┬────────┬───────────┐
│ all_measure_count ┆ measure_signature                                       ┆ count  ┆ pct       │
│ ---               ┆ ---                                                     ┆ ---    ┆ ---       │
│ u32               ┆ str                                                     ┆ u32    ┆ f64       │
╞═══════════════════╪═════════════════════════════════════════════════════════╪════════╪═══════════╡
│ 1                 ┆ RR                                                      ┆ 733800 ┆ 62.594045 │
│ 5                 ┆ RR,TN,TX,TNTXM,TAMPLI                                   ┆ 280908 ┆ 23.961799 │
│ 9                 ┆ RR,TN,HTN,TX,HTX,TM,TNTXM,TAMPLI,DG                     ┆ 81036  ┆ 6.912471  │
│ 19                ┆ RR,TN,HTN,TX,HTX,TM,TNTXM,TAMPLI,DG,FFM,FXY,DXY,HXY,FXI ┆ 23138  ┆ 1.9737    

#### Value / quality consistency

In [ ]:
rtv_all_quality_audit = dwl.quality_pair_audit_lf(lf_rtv_wide_all, dwl.RR_T_VENT_ALL_PARAMS)

with pl.Config(tbl_rows=100):
    print(rtv_all_quality_audit)

shape: (52, 2)
┌──────────────────────────────┬───────┐
│ line_type                    ┆ count │
│ ---                          ┆ ---   │
│ str                          ┆ u32   │
╞══════════════════════════════╪═══════╡
│ RR_value_without_quality     ┆ 0     │
│ RR_quality_without_value     ┆ 69    │
│ TN_value_without_quality     ┆ 0     │
│ TN_quality_without_value     ┆ 1711  │
│ HTN_value_without_quality    ┆ 0     │
│ HTN_quality_without_value    ┆ 2066  │
│ TX_value_without_quality     ┆ 0     │
│ TX_quality_without_value     ┆ 1705  │
│ HTX_value_without_quality    ┆ 0     │
│ HTX_quality_without_value    ┆ 2082  │
│ TM_value_without_quality     ┆ 0     │
│ TM_quality_without_value     ┆ 1714  │
│ TNTXM_value_without_quality  ┆ 0     │
│ TNTXM_quality_without_value  ┆ 1     │
│ TAMPLI_value_without_quality ┆ 0     │
│ TAMPLI_quality_without_value ┆ 1     │
│ TNSOL_value_without_quality  ┆ 0     │
│ TNSOL_quality_without_value  ┆ 24    │
│ TN50_value_without_quality   ┆ 0     │
│

In [ ]:
orphan_quality_audit_rtv_all = pl.DataFrame()
for param in dwl.RR_T_VENT_ALL_PARAMS:
    value_col = param.lower()
    quality_col = f"q{value_col}"

    orphan_quality_audit_rtv_all = pl.concat([
        orphan_quality_audit_rtv_all,
        lf_rtv_wide_all
        .filter(
            pl.col(value_col).is_null()
            & pl.col(quality_col).is_not_null()
        )
        .with_columns(
            quality_column=pl.lit(quality_col),
        )
        .group_by("quality_column", quality_col)
        .len()
        .sort(quality_col)
        .rename({quality_col: "quality_code_value"})
        .collect()
    ])

with pl.Config(tbl_rows=100):
    print(orphan_quality_audit_rtv_all)

shape: (39, 3)
┌────────────────┬────────────────────┬──────┐
│ quality_column ┆ quality_code_value ┆ len  │
│ ---            ┆ ---                ┆ ---  │
│ str            ┆ u8                 ┆ u32  │
╞════════════════╪════════════════════╪══════╡
│ qrr            ┆ 1                  ┆ 69   │
│ qtn            ┆ 1                  ┆ 1683 │
│ qtn            ┆ 9                  ┆ 28   │
│ qhtn           ┆ 1                  ┆ 2032 │
│ qhtn           ┆ 9                  ┆ 34   │
│ qtx            ┆ 1                  ┆ 1687 │
│ qtx            ┆ 9                  ┆ 18   │
│ qhtx           ┆ 1                  ┆ 2051 │
│ qhtx           ┆ 9                  ┆ 31   │
│ qtm            ┆ 1                  ┆ 1711 │
│ qtm            ┆ 9                  ┆ 3    │
│ qtntxm         ┆ 1                  ┆ 1    │
│ qtampli        ┆ 1                  ┆ 1    │
│ qtnsol         ┆ 1                  ┆ 15   │
│ qtnsol         ┆ 9                  ┆ 9    │
│ qdg            ┆ 1                  ┆ 1751 

### All autres-parametres parameters (41 parameters)

#### Measurement density

In [ ]:
lf_ap_wide = dwl.add_measure_count_lf(lf_ap_wide, dwl.AUTRES_PARAMETRES_ALL_PARAMS, "all_measure_count")
expected_long_core_rows_ap = (
    lf_ap_wide
    .select(pl.col("all_measure_count").sum())
    .collect()
    .item()
)
mean_measures_per_wide_row_ap = (
    expected_long_core_rows_ap / df_ap_wide.height
)
measure_density_ap = (
    expected_long_core_rows_ap
    / (df_ap_wide.height * len(dwl.AUTRES_PARAMETRES_ALL_PARAMS))
)

with pl.Config(tbl_cols=20, tbl_rows=100):
    print("autres-parametres all parameters:")
    print(f"height: {df_ap_wide.height}")
    print(f"expected_long_core_rows: {expected_long_core_rows_ap}")
    print(f"mean_measures_per_wide_row: {mean_measures_per_wide_row_ap}")
    print(f"measure_density: {measure_density_ap}")

    print(
        lf_ap_wide
        .group_by("all_measure_count")
        .agg(
            count=pl.len(),
            pct=pl.len() / df_ap_wide.height * 100
        )
        .sort("all_measure_count")
        .collect()
    )

autres-parametres all parameters:
height: 565558
expected_long_core_rows: 2148378
mean_measures_per_wide_row: 3.7986873141216284
measure_density: 0.09265091010052752
shape: (30, 3)
┌───────────────────┬────────┬───────────┐
│ all_measure_count ┆ count  ┆ pct       │
│ ---               ┆ ---    ┆ ---       │
│ u32               ┆ u32    ┆ f64       │
╞═══════════════════╪════════╪═══════════╡
│ 0                 ┆ 370    ┆ 0.065422  │
│ 1                 ┆ 122836 ┆ 21.719435 │
│ 2                 ┆ 21626  ┆ 3.823834  │
│ 3                 ┆ 278177 ┆ 49.18629  │
│ 4                 ┆ 71335  ┆ 12.613207 │
│ 5                 ┆ 1198   ┆ 0.211826  │
│ 6                 ┆ 1330   ┆ 0.235166  │
│ 7                 ┆ 1741   ┆ 0.307838  │
│ 8                 ┆ 794    ┆ 0.140392  │
│ 9                 ┆ 39317  ┆ 6.951895  │
│ 10                ┆ 306    ┆ 0.054106  │
│ 11                ┆ 7693   ┆ 1.36025   │
│ 12                ┆ 1256   ┆ 0.222082  │
│ 13                ┆ 2783   ┆ 0.49208   │
│ 

#### Recurring parameter signatures

In [ ]:
lf_ap_wide = lf_ap_wide.with_columns(
    dwl.measure_signature_expr(dwl.AUTRES_PARAMETRES_ALL_PARAMS)
    .alias("measure_signature")
)

patterns_ap = (
    lf_ap_wide
    .group_by(
        "all_measure_count",
        "measure_signature",
    )
    .agg(
        count=pl.len(),
    )
    .with_columns(
        pct=pl.col("count") / df_ap_wide.height * 100
    )
    .sort("count", descending=True)
    .collect()
)

patterns_ap_head = patterns_ap.head(30)
print(f"patterns_ap.shape: {patterns_ap.shape}")
print(f"patterns_ap, sum pct: {patterns_ap.select("pct").sum().item()}")
with pl.Config(tbl_cols=20, tbl_rows=100, fmt_str_lengths=300):   
    print(patterns_ap_head)


patterns_ap.shape: (635, 4)
patterns_ap, sum pct: 99.99999999999996
shape: (30, 4)
┌───────────────────┬─────────────────────────────────────────────────────────┬────────┬───────────┐
│ all_measure_count ┆ measure_signature                                       ┆ count  ┆ pct       │
│ ---               ┆ ---                                                     ┆ ---    ┆ ---       │
│ u32               ┆ str                                                     ┆ u32    ┆ f64       │
╞═══════════════════╪═════════════════════════════════════════════════════════╪════════╪═══════════╡
│ 3                 ┆ BROU,ORAG,GRELE                                         ┆ 270391 ┆ 47.809597 │
│ 1                 ┆ ETPGRILLE                                               ┆ 114089 ┆ 20.17282  │
│ 4                 ┆ ETPGRILLE,BROU,ORAG,GRELE                               ┆ 67175  ┆ 11.87765  │
│ 9                 ┆ UN,HUN,UX,HUX,UM,DHUMI40,DHUMI80,TSVM,ETPGRILLE         ┆ 34402  ┆ 6.082842  │
│ 2     

#### Value / quality consistency

In [ ]:
ap_quality_audit = dwl.quality_pair_audit_lf(lf_ap_wide, dwl.AUTRES_PARAMETRES_ALL_PARAMS)

with pl.Config(tbl_rows=100):
    print(ap_quality_audit)

shape: (82, 2)
┌─────────────────────────────────┬───────┐
│ line_type                       ┆ count │
│ ---                             ┆ ---   │
│ str                             ┆ u32   │
╞═════════════════════════════════╪═══════╡
│ DHUMEC_value_without_quality    ┆ 0     │
│ DHUMEC_quality_without_value    ┆ 151   │
│ PMERM_value_without_quality     ┆ 0     │
│ PMERM_quality_without_value     ┆ 0     │
│ PMERMIN_value_without_quality   ┆ 0     │
│ PMERMIN_quality_without_value   ┆ 2     │
│ INST_value_without_quality      ┆ 0     │
│ INST_quality_without_value      ┆ 0     │
│ GLOT_value_without_quality      ┆ 0     │
│ GLOT_quality_without_value      ┆ 5     │
│ DIFT_value_without_quality      ┆ 0     │
│ DIFT_quality_without_value      ┆ 0     │
│ DIRT_value_without_quality      ┆ 0     │
│ DIRT_quality_without_value      ┆ 0     │
│ INFRART_value_without_quality   ┆ 0     │
│ INFRART_quality_without_value   ┆ 0     │
│ UV_value_without_quality        ┆ 0     │
│ UV_quality_with

In [ ]:
orphan_quality_audit_ap = pl.DataFrame()
for param in dwl.AUTRES_PARAMETRES_ALL_PARAMS:
    value_col = param.lower()
    quality_col = f"q{value_col}"

    orphan_quality_audit_ap = pl.concat([
        orphan_quality_audit_ap,
        lf_ap_wide
        .filter(
            pl.col(value_col).is_null()
            & pl.col(quality_col).is_not_null()
        )
        .with_columns(
            quality_column=pl.lit(quality_col),
        )
        .group_by("quality_column", quality_col)
        .len()
        .sort(quality_col)
        .rename({quality_col: "quality_code_value"})
        .collect()
    ])

with pl.Config(tbl_rows=100):
    print(orphan_quality_audit_ap)

shape: (48, 3)
┌────────────────┬────────────────────┬───────┐
│ quality_column ┆ quality_code_value ┆ len   │
│ ---            ┆ ---                ┆ ---   │
│ str            ┆ u8                 ┆ u32   │
╞════════════════╪════════════════════╪═══════╡
│ qdhumec        ┆ 1                  ┆ 151   │
│ qpmermin       ┆ 1                  ┆ 1     │
│ qpmermin       ┆ 9                  ┆ 1     │
│ qglot          ┆ 1                  ┆ 5     │
│ qun            ┆ 1                  ┆ 44    │
│ qun            ┆ 9                  ┆ 1     │
│ qhun           ┆ 1                  ┆ 82    │
│ qhun           ┆ 9                  ┆ 3     │
│ qux            ┆ 1                  ┆ 27    │
│ qux            ┆ 9                  ┆ 1     │
│ qhux           ┆ 1                  ┆ 45    │
│ qhux           ┆ 9                  ┆ 1     │
│ qum            ┆ 1                  ┆ 28    │
│ qum            ┆ 9                  ┆ 2     │
│ qdhumi40       ┆ 1                  ┆ 54    │
│ qdhumi40       ┆ 9     